# 00 · Fallback Gold — Build Complete Gold Model

**Trainer use only.** Run this notebook at the end of Day 1 (after Workshop 1)
or at the start of Day 2 (after Workshop 2 if a participant is stuck).

Set TARGET_CATALOG to the participant's catalog name (e.g., retailhub_jan_kowalski).
Leave empty to build for the current user's catalog.

In [ ]:
dbutils.widgets.text("TARGET_CATALOG", "", "Target Catalog (leave empty for current user)")

In [ ]:
import re

CATALOG_PREFIX = "retailhub"

target = dbutils.widgets.get("TARGET_CATALOG").strip()
if target:
    CATALOG = target
else:
    current_user = spark.sql("SELECT current_user()").collect()[0][0]
    local_part = current_user.lower().split("@")[0]
    if "trainer" in local_part or "trener" in local_part or local_part == "krzysztof.burejza":
        CATALOG = "retailhub_trainer"
    else:
        slug = re.sub(r"[^a-zA-Z0-9]", "_", local_part).lower()
        CATALOG = f"{CATALOG_PREFIX}_{slug}"

GOLD = f"{CATALOG}.gold"
print(f"Building Gold model for catalog: {CATALOG}")
print(f"Gold schema: {GOLD}")

In [ ]:
spark.sql(f"USE CATALOG {CATALOG}")

required_silver = ["silver.customers", "silver.products", "silver.sales_orders", "silver.order_lines"]
missing = [t for t in required_silver if not spark.catalog.tableExists(t)]
if missing:
    raise Exception(f"Silver tables missing in {CATALOG}: {missing}. Run generate_training_dataset first.")
print(f"[OK] Silver tables found in {CATALOG}")

## Building Gold Objects (9 total)
Each cell builds one object and prints [OK] when done.

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {GOLD}.dim_date AS
    SELECT DISTINCT
      order_date                         AS date_key,
      YEAR(order_date)                   AS year,
      QUARTER(order_date)                AS quarter,
      MONTH(order_date)                  AS month,
      DATE_FORMAT(order_date, 'yyyy-MM') AS year_month,
      DAYOFWEEK(order_date)              AS day_of_week
    FROM silver.sales_orders
    WHERE order_date IS NOT NULL
    ORDER BY date_key
""")
n = spark.sql(f"SELECT COUNT(*) FROM {GOLD}.dim_date").collect()[0][0]
print(f"[OK] dim_date — {n} rows")

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {GOLD}.dim_customer AS
    SELECT customer_id, customer_name, segment, region AS customer_region
    FROM silver.customers
""")
n = spark.sql(f"SELECT COUNT(*) FROM {GOLD}.dim_customer").collect()[0][0]
print(f"[OK] dim_customer — {n} rows")

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {GOLD}.dim_product AS
    SELECT product_id, product_name, category, subcategory, unit_cost
    FROM silver.products
""")
n = spark.sql(f"SELECT COUNT(*) FROM {GOLD}.dim_product").collect()[0][0]
print(f"[OK] dim_product — {n} rows")

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {GOLD}.fact_sales AS
    SELECT
      l.line_id, l.order_id, o.customer_id, l.product_id,
      o.order_date, o.status, o.channel,
      l.quantity, l.unit_price, p.unit_cost,
      CASE WHEN o.status = 'Completed' THEN true ELSE false END AS is_completed,
      l.quantity * l.unit_price                                  AS line_revenue,
      l.quantity * (l.unit_price - p.unit_cost)                 AS line_margin
    FROM silver.order_lines l
    JOIN silver.sales_orders o ON l.order_id  = o.order_id
    LEFT JOIN silver.products p ON l.product_id = p.product_id
""")
n = spark.sql(f"SELECT COUNT(*) FROM {GOLD}.fact_sales").collect()[0][0]
print(f"[OK] fact_sales — {n} rows")

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {GOLD}.fact_sales_dashboard AS
    SELECT
      f.line_id, f.order_id, f.order_date, f.status, f.channel,
      f.quantity, f.unit_price, f.unit_cost,
      f.is_completed, f.line_revenue, f.line_margin,
      d.year, d.quarter, d.month, d.year_month, d.day_of_week,
      c.customer_name, c.segment, c.customer_region,
      p.product_name, p.category, p.subcategory
    FROM {GOLD}.fact_sales f
    LEFT JOIN {GOLD}.dim_date     d ON f.order_date  = d.date_key
    LEFT JOIN {GOLD}.dim_customer c ON f.customer_id = c.customer_id
    LEFT JOIN {GOLD}.dim_product  p ON f.product_id  = p.product_id
""")
n = spark.sql(f"SELECT COUNT(*) FROM {GOLD}.fact_sales_dashboard").collect()[0][0]
print(f"[OK] fact_sales_dashboard — {n} rows")

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {GOLD}.kpi_daily AS
    SELECT
      order_date,
      COUNT(DISTINCT CASE WHEN is_completed THEN order_id END)           AS completed_orders,
      ROUND(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 2) AS revenue,
      ROUND(SUM(CASE WHEN is_completed THEN line_margin  ELSE 0 END), 2) AS gross_margin,
      ROUND(
        100.0 * COUNT(DISTINCT CASE WHEN status = 'Returned' THEN order_id END)
              / NULLIF(COUNT(DISTINCT order_id), 0), 2
      ) AS return_rate_pct
    FROM {GOLD}.fact_sales_dashboard
    GROUP BY order_date
""")
n = spark.sql(f"SELECT COUNT(*) FROM {GOLD}.kpi_daily").collect()[0][0]
print(f"[OK] kpi_daily — {n} rows")

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {GOLD}.kpi_monthly AS
    SELECT
      d.year_month,
      SUM(k.completed_orders)          AS completed_orders,
      ROUND(SUM(k.revenue), 2)         AS revenue,
      ROUND(SUM(k.gross_margin), 2)    AS gross_margin,
      ROUND(AVG(k.return_rate_pct), 2) AS return_rate_pct
    FROM {GOLD}.kpi_daily k
    JOIN {GOLD}.dim_date d ON k.order_date = d.date_key
    GROUP BY d.year_month
    ORDER BY d.year_month
""")
n = spark.sql(f"SELECT COUNT(*) FROM {GOLD}.kpi_monthly").collect()[0][0]
print(f"[OK] kpi_monthly — {n} rows")

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {GOLD}.fact_sales_dashboard_monthly AS
    SELECT
      year_month, customer_region, category, channel,
      COUNT(DISTINCT CASE WHEN is_completed THEN order_id END)           AS completed_orders,
      ROUND(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 2) AS revenue,
      ROUND(SUM(CASE WHEN is_completed THEN line_margin  ELSE 0 END), 2) AS gross_margin
    FROM {GOLD}.fact_sales_dashboard
    GROUP BY year_month, customer_region, category, channel
""")
n = spark.sql(f"SELECT COUNT(*) FROM {GOLD}.fact_sales_dashboard_monthly").collect()[0][0]
print(f"[OK] fact_sales_dashboard_monthly — {n} rows")

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE VIEW {GOLD}.v_fact_sales_incremental AS
    SELECT *
    FROM {GOLD}.fact_sales_dashboard
    WHERE order_date >= ADD_MONTHS(CURRENT_DATE(), -24)
""")
n = spark.sql(f"SELECT COUNT(*) FROM {GOLD}.v_fact_sales_incremental").collect()[0][0]
print(f"[OK] v_fact_sales_incremental — {n} rows (last 24 months)")

In [ ]:
objects = [
    f"{GOLD}.dim_date", f"{GOLD}.dim_customer", f"{GOLD}.dim_product",
    f"{GOLD}.fact_sales", f"{GOLD}.fact_sales_dashboard",
    f"{GOLD}.kpi_daily", f"{GOLD}.kpi_monthly",
    f"{GOLD}.fact_sales_dashboard_monthly", f"{GOLD}.v_fact_sales_incremental",
]
missing = [o for o in objects if not spark.catalog.tableExists(o)]
if missing:
    print("[FAILED] Missing:", missing)
else:
    print(f"\n[OK] Gold model complete for: {CATALOG}")
    print(f"     9 objects created. Participant can now run day2_01_powerbi_semantic_model.")